# LeNet5 / MNIST: a slow walk through the 1M-skeleton sticky PDMP samples

Two runs, both LeNet5 on an MNIST subset, both sampled for **1,000,000
skeleton events** across 100 stages:

- `grid_sticky_zigzag.pt`  -- Sticky Zig-Zag
- `grid_sticky_boomerang.pt` -- Sticky Boomerang

The point of this notebook is *not* to rush to a headline number. We go in
order:

1. **What is actually in the files?** Every key, every shape, every dtype,
   value ranges, what is present and what is missing.
2. **The input-to-first-hidden weights (`conv1`).** MNIST digits sit in a
   fixed frame with an always-black border. `conv1` is the only layer that
   touches raw pixels. Do the samples put many exact zeros on the weights
   that only ever look at the border? Does per-draw uncertainty avoid the
   border?
3. Only then: sparsity over the path, displacement from the MAP reference,
   the conv1 activation-uncertainty figure (kept from `sticky_cnn_eval.ipynb`),
   and predictive accuracy / uncertainty.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
%matplotlib inline

if Path.cwd().name == "notebooks":
    os.chdir("..")

plt.rcParams.update({
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.3,
    "font.size":          11,
})

RUN_DIR = Path("results/grid/mnist_cnn/split_00")
RUN_SPECS = [
    ("zigzag",    "grid_sticky_zigzag.pt"),
    ("boomerang", "grid_sticky_boomerang.pt"),
]
RUN_DISPLAY_NAME = {"zigzag": "Sticky Zig-Zag", "boomerang": "Sticky Boomerang"}
print("cwd:", Path.cwd())


## 1. What is in the files?

### 1a. File-level: size, top-level type, every key

`torch.load(..., weights_only=False)` because these checkpoints hold Python
scalars and lists next to the tensors. For each key we print: tensor shape /
dtype / min / max / a near-zero fraction for the big ones, or the raw value
for scalars, or length + head for lists.


In [ ]:
def describe_ckpt(ck):
    rows = []
    for k, v in ck.items():
        if torch.is_tensor(v):
            near0 = float((v.abs() < 1e-8).float().mean()) if v.dtype.is_floating_point else np.nan
            rows.append({
                "key": k, "kind": "tensor", "shape": str(tuple(v.shape)),
                "dtype": str(v.dtype),
                "min": float(v.min()) if v.numel() else np.nan,
                "max": float(v.max()) if v.numel() else np.nan,
                "frac |.|<1e-8": near0,
            })
        elif isinstance(v, (list, tuple)):
            rows.append({"key": k, "kind": type(v).__name__, "shape": f"len={len(v)}",
                         "dtype": "", "min": np.nan, "max": np.nan, "frac |.|<1e-8": np.nan,
                         "value/head": repr(v[:3])})
        elif isinstance(v, dict):
            rows.append({"key": k, "kind": "dict", "shape": f"{len(v)} keys",
                         "dtype": "", "min": np.nan, "max": np.nan, "frac |.|<1e-8": np.nan,
                         "value/head": repr(list(v.keys())[:8])})
        else:
            rows.append({"key": k, "kind": type(v).__name__, "shape": "", "dtype": "",
                         "min": np.nan, "max": np.nan, "frac |.|<1e-8": np.nan,
                         "value/head": repr(v)})
    return pd.DataFrame(rows).set_index("key")


runs = {}
for label, fname in RUN_SPECS:
    path = RUN_DIR / fname
    if not path.exists():
        print(f"[{label}] MISSING -- {path}")
        continue
    size_gb = path.stat().st_size / 1e9
    ck = torch.load(path, map_location="cpu", weights_only=False)
    runs[label] = {"ckpt": ck, "D": int(ck["x_ref"].shape[0]), "path": path, "size_gb": size_gb}
    print("=" * 78)
    print(f"[{label}]  {fname}   {size_gb:.2f} GB on disk   top-level type: {type(ck).__name__}")
    print("=" * 78)
    display(describe_ckpt(ck))

assert runs, f"No run files under {RUN_DIR}"


### 1b. What do we have, and what is missing?

Plain-language summary of the important fields, plus an explicit list of what
is *not* in these checkpoints -- so later sections do not silently assume it.


In [ ]:
for label, r in runs.items():
    ck = r["ckpt"]
    n_draws, D = ck["samples"].shape
    print(f"[{RUN_DISPLAY_NAME[label]}]")
    print(f"  sampler              : {ck['sampler']}")
    print(f"  architecture         : LeNet5  (activation={ck['activation']}, pool={ck['pool']})")
    print(f"  D (params)           : {D}")
    print(f"  skeleton events run  : {ck['n_events']:,}")
    print(f"  resampled draws saved: {n_draws:,}   -> samples tensor is [{n_draws}, {D}]")
    print(f"  wall time            : {ck['elapsed_sec']:.1f} s  ({ck['elapsed_sec']/3600:.2f} h)")
    print(f"  gradient evals       : {ck['gradient_evals']:,}")
    print(f"  bound_violations     : {ck['bound_violations']}")
    print(f"  prune_frac (t=0)     : {ck['prune_frac']:.4f}   (fraction cold-start frozen)")
    print(f"  sparsity_frac (saved): {ck['sparsity_frac']:.4f}   (near-zero fraction over resampled draws)")
    print(f"  test_accuracy (saved): {ck['test_accuracy']:.4f}")
    print(f"  cold_start_mask      : bool[{ck['cold_start_mask'].shape[0]}], "
          f"{int(ck['cold_start_mask'].sum())} True ({100*ck['cold_start_mask'].float().mean():.2f}%)")
    print(f"  grid_t_max_log       : list, len {len(ck['grid_t_max_log'])}  "
          f"(one entry per _grid_bound call)")
    diag = ck.get("diagnostics")
    print(f"  diagnostics          : {type(diag).__name__}"
          + ("" if diag is None else f"  ({len(diag)} rows)"))
    print()

print("NOT in these checkpoints (so the corresponding analysis is unavailable):")
print("  - per-iteration diagnostics list (diagnostics is None for both runs)")
chunk_dir = RUN_DIR.parent / "chunks" / "mnist_cnn" / "split_00"
print(f"  - flushed diag_*.pt chunk files: chunk dir {chunk_dir} exists = {chunk_dir.exists()}")
print("  => no per-coordinate bounce attribution, no freeze/thaw/bounce event mix,")
print("     no per-iteration max_ratio. Section on movement uses only samples +")
print("     cold_start_mask (never-left-zero vs moved), which needs no diagnostics.")
print("  - the full skeleton path: only 10,000 resampled draws are stored, not all 1e6 events.")


### 1c. The `samples` tensor up close

Dimensionality, dtype, memory, and the marginal spread of a few coordinates.
`samples[i]` is one full parameter vector (one posterior draw); `samples[:, j]`
is the marginal path of coordinate `j` across the 10k resampled draws.


In [ ]:
for label, r in runs.items():
    s = r["ckpt"]["samples"]
    print(f"[{RUN_DISPLAY_NAME[label]}] samples: shape={tuple(s.shape)} dtype={s.dtype} "
          f"contiguous={s.is_contiguous()} mem={s.element_size()*s.nelement()/1e6:.1f} MB")
    exact_zero = (s == 0)
    print(f"    exact zeros: {int(exact_zero.sum()):,} / {s.nelement():,} "
          f"({100*exact_zero.float().mean():.2f}%)   "
          f"rows with >=1 exact zero: {int(exact_zero.any(1).sum())}/{s.shape[0]}")
    per_draw_near0 = (s.abs() < 1e-8).float().mean(1)
    print(f"    per-draw near-zero (|.|<1e-8) fraction: "
          f"min={per_draw_near0.min().item():.4f}  max={per_draw_near0.max().item():.4f}")
    # marginal std across draws, summarised
    col_std = s.std(0)
    print(f"    per-coordinate std across draws: "
          f"median={col_std.median().item():.4g}  p95={col_std.quantile(0.95).item():.4g}  "
          f"max={col_std.max().item():.4g}")
    print(f"    fully-static coordinates (std==0 across all draws): "
          f"{int((col_std == 0).sum()):,} / {s.shape[1]:,}")
    print()


## 2. Layer map: unflatten coordinates back to `(layer, position)`

Every coordinate `0..61705` belongs to one LeNet5 parameter tensor. This
mapping is fixed by `module.named_parameters()` order in
`sazz/gpu_friendly/models/neural_networks.py` and is what lets us pull out
"the `conv1.weight` coordinates" below.


In [ ]:
LENET5_SHAPES = [
    ("conv1.weight", (6, 1, 5, 5)), ("conv1.bias", (6,)),
    ("conv2.weight", (16, 6, 5, 5)), ("conv2.bias", (16,)),
    ("fc1.weight", (120, 400)), ("fc1.bias", (120,)),
    ("fc2.weight", (84, 120)), ("fc2.bias", (84,)),
    ("fc3.weight", (10, 84)), ("fc3.bias", (10,)),
]
D_EXPECTED = sum(int(np.prod(s)) for _, s in LENET5_SHAPES)
assert D_EXPECTED == 61706, D_EXPECTED


def build_layer_index(D):
    assert D == D_EXPECTED, f"D={D} != LeNet5 {D_EXPECTED}"
    rows, idx = [], 0
    for name, shape in LENET5_SHAPES:
        n = int(np.prod(shape))
        is_bias = len(shape) == 1
        for local_i in range(n):
            pos = (local_i,) if is_bias else tuple(int(x) for x in np.unravel_index(local_i, shape))
            rows.append({"coord": idx, "layer": name, "is_bias": is_bias, "shape": shape, "pos": pos})
            idx += 1
    return pd.DataFrame(rows).set_index("coord")


layer_idx_dfs = {label: build_layer_index(r["D"]) for label, r in runs.items()}
ldf0 = next(iter(layer_idx_dfs.values()))
display(ldf0.groupby("layer", sort=False).agg(n=("layer", "size"),
                                              first_coord=("is_bias", lambda s: s.index.min()),
                                              last_coord=("is_bias", lambda s: s.index.max())))


## 3. `conv1`: the input-to-first-hidden weights and the black border

MNIST digits are centred in a 28x28 frame; LeNet5 zero-pads to 32x32
(`F.pad(x, [2,2,2,2])`), then `conv1 = Conv2d(1, 6, kernel_size=5)`. The outer
ring of pixels is (almost) always zero, so a `conv1` weight whose 5x5 receptive
field only ever lands on border pixels sees no signal, whatever the digit.

We ask three concrete questions of the posterior draws:

- **3a.** For each of the 6 filters, what does the *posterior-mean* 5x5 kernel
  look like, and what is the exact-zero rate per kernel tap across draws?
- **3b.** For each of the 6x28x28 `conv1` *output positions*, compare the
  training-set input-patch variance it reads against the across-draw std of
  the posterior activation there. Do the border output positions (which read
  mostly dead zero-padding) stay quiet across draws?
- **3c.** The per-pixel picture: push the mean kernel over a blank frame and a
  real digit; where does the border sit.


In [ ]:
from torchvision import datasets, transforms

_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
_mnist_train = datasets.MNIST("datasets", train=True, download=True, transform=_transform)
_rng = np.random.default_rng(0)
_sample_idx = _rng.choice(len(_mnist_train), size=2000, replace=False)

# Per-pixel mean and variance of the padded 32x32 image over the sample.
_imgs = torch.stack([_mnist_train[int(i)][0].squeeze(0) for i in _sample_idx])  # [N,28,28]
_imgs_padded = torch.nn.functional.pad(_imgs, [2, 2, 2, 2])                     # [N,32,32]
pix_mean = _imgs_padded.mean(0)   # [32,32]
pix_var = _imgs_padded.var(0)     # [32,32]

fig, ax = plt.subplots(1, 3, figsize=(11, 3.6))
im0 = ax[0].imshow(pix_mean, cmap="gray");   ax[0].set_title("padded MNIST: per-pixel mean")
im1 = ax[1].imshow(pix_var, cmap="magma");   ax[1].set_title("padded MNIST: per-pixel variance")
ax[2].imshow(pix_var < 1e-12, cmap="gray");   ax[2].set_title("variance < 1e-12  (dead pixels)")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
fig.colorbar(im0, ax=ax[0], fraction=0.046); fig.colorbar(im1, ax=ax[1], fraction=0.046)
fig.tight_layout(); plt.show()
print(f"dead (var<1e-4) padded pixels: {int((pix_var < 1e-4).sum())}/1024")


### 3a. Posterior-mean `conv1` kernels and per-tap exact-zero rate

`conv1.weight` is `[6, 1, 5, 5]` -> 150 coordinates. For each run we take all
10k draws, reshape to `[10000, 6, 5, 5]`, and show the mean kernel and the
fraction of draws where each tap is exactly zero.


In [ ]:
for label, r in runs.items():
    ldf = layer_idx_dfs[label]
    w_coords = torch.as_tensor(ldf.index[ldf["layer"] == "conv1.weight"].to_numpy())
    s = r["ckpt"]["samples"][:, w_coords].view(-1, 6, 5, 5).float()  # [10000,6,5,5]

    mean_k = s.mean(0)                       # [6,5,5]
    zero_rate = (s == 0).float().mean(0)     # [6,5,5]
    std_k = s.std(0)                         # [6,5,5]

    fig, axes = plt.subplots(3, 6, figsize=(13, 6.6))
    vlim = mean_k.abs().max()
    for c in range(6):
        axes[0, c].imshow(mean_k[c], cmap="RdBu_r", vmin=-vlim, vmax=vlim)
        axes[0, c].set_title(f"filter {c}\nmean kernel", fontsize=9)
        axes[1, c].imshow(zero_rate[c], cmap="Greys", vmin=0, vmax=1)
        axes[1, c].set_title(f"exact-zero rate\nmean {zero_rate[c].mean():.2f}", fontsize=9)
        axes[2, c].imshow(std_k[c], cmap="viridis")
        axes[2, c].set_title(f"std across draws\nmean {std_k[c].mean():.3f}", fontsize=9)
    for a in axes.flat:
        a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f"{RUN_DISPLAY_NAME[label]}: conv1.weight -- mean kernel, exact-zero rate, per-tap std",
                 fontsize=12)
    fig.tight_layout(); plt.show()

    print(f"[{RUN_DISPLAY_NAME[label]}] conv1.weight: "
          f"overall exact-zero rate {(s == 0).float().mean():.3f}, "
          f"taps that are zero in EVERY draw: {int((s == 0).all(0).sum())}/150, "
          f"taps that are never zero: {int((s != 0).all(0).sum())}/150")


### 3b. Border **output positions**: does posterior uncertainty avoid them?

Averaging a single 5x5 tap over all 28x28 conv positions washes the border
out -- every tap ends up reading a near-identical mix of pixels (checked
below: receptive variance is ~0.71 for all 25 taps). The border signal lives
in *where the kernel sits*, not in *which tap*.

So we work per **output position** instead. `conv1` produces a `[6, 28, 28]`
map; output position `(oh, ow)` reads the 5x5 padded-image patch anchored at
`(oh, ow)`. For each of the 28x28 positions:

- **input-patch variance**: mean training-set `pix_var` over that 5x5 patch.
  Positions near the frame edge read mostly dead zero-padding -> low.
- push all 10k posterior draws of `conv1` weights+bias through `conv1` on one
  real digit, and record the **across-draw std** and **near-zero rate** of the
  activation at that position.

If stickiness concentrates uncertainty where there is signal, border output
positions (low input-patch variance) should have **low across-draw std**.


In [ ]:
# 3b.i -- confirm the per-tap receptive variance really is flat (why the
# per-tap framing fails), then do the per-output-position analysis.
recept_var_tap = torch.zeros(5, 5)
for kh in range(5):
    for kw in range(5):
        recept_var_tap[kh, kw] = pix_var[kh:kh + 28, kw:kw + 28].mean()
print("per-tap receptive variance (5x5) -- essentially constant, hence the reframe:")
print(np.array2string(recept_var_tap.numpy(), precision=3, suppress_small=True))

# input-patch variance per conv1 OUTPUT position (oh, ow), oh/ow in 0..27:
# patch = padded[oh:oh+5, ow:ow+5]
patch_var = torch.zeros(28, 28)
patch_meanabs = torch.zeros(28, 28)
for oh in range(28):
    for ow in range(28):
        patch_var[oh, ow] = pix_var[oh:oh + 5, ow:ow + 5].mean()
        patch_meanabs[oh, ow] = pix_mean[oh:oh + 5, ow:ow + 5].abs().mean()

_digit_idx = int(_sample_idx[5])
_digit_img = _mnist_train[_digit_idx][0].squeeze(0)                                   # [28,28]
_digit_padded = torch.nn.functional.pad(_digit_img, [2, 2, 2, 2]).view(1, 1, 32, 32)  # [1,1,32,32]

pos_rows = []
for label, r in runs.items():
    ldf = layer_idx_dfs[label]
    w_coords = torch.as_tensor(ldf.index[ldf["layer"] == "conv1.weight"].to_numpy())
    b_coords = torch.as_tensor(ldf.index[ldf["layer"] == "conv1.bias"].to_numpy())
    s = r["ckpt"]["samples"]

    acts = []  # [n_draws, 6, 28, 28]
    with torch.no_grad():
        for i in range(s.shape[0]):
            w = s[i, w_coords].view(6, 1, 5, 5).float()
            b = s[i, b_coords].float()
            acts.append(torch.nn.functional.conv2d(_digit_padded.float(), w, b).squeeze(0))
    acts = torch.stack(acts)

    act_std = acts.std(0)                       # [6,28,28] across-draw std per position
    act_near0 = (acts.abs() < 1e-6).float().mean(0)  # [6,28,28] near-zero rate per position
    act_std_pos = act_std.mean(0)              # [28,28] averaged over the 6 filters
    act_near0_pos = act_near0.mean(0)

    # figure: input-patch variance vs across-draw activation std, as maps
    fig, ax = plt.subplots(1, 4, figsize=(15, 3.8))
    m0 = ax[0].imshow(patch_var, cmap="magma");        ax[0].set_title("input-patch variance\n(per output pos)", fontsize=9)
    m1 = ax[1].imshow(patch_var < 0.05, cmap="gray");  ax[1].set_title("border positions\n(patch var < 0.05)", fontsize=9)
    m2 = ax[2].imshow(act_std_pos, cmap="viridis");    ax[2].set_title(f"{RUN_DISPLAY_NAME[label]}\nacross-draw activation std", fontsize=9)
    m3 = ax[3].imshow(act_near0_pos, cmap="cividis");  ax[3].set_title("activation near-zero rate\nacross draws", fontsize=9)
    for a, mm in zip(ax, (m0, m1, m2, m3)):
        a.set_xticks([]); a.set_yticks([]); fig.colorbar(mm, ax=a, fraction=0.046)
    fig.tight_layout(); plt.show()

    # scatter + border/interior split
    border = (patch_var < 0.05)
    pv = patch_var.flatten().numpy()
    sd = act_std_pos.flatten().numpy()
    n0 = act_near0_pos.flatten().numpy()
    bmask = border.flatten().numpy()

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    ax[0].scatter(pv[~bmask], sd[~bmask], s=10, alpha=0.5, label="interior pos")
    ax[0].scatter(pv[bmask], sd[bmask], s=10, alpha=0.6, label="border pos", color="#c44e52")
    ax[0].set(xlabel="input-patch variance", ylabel="across-draw activation std",
              title=f"{RUN_DISPLAY_NAME[label]}: uncertainty vs input-patch variance")
    ax[0].legend(fontsize=8)
    ax[1].hist(sd[~bmask], bins=40, histtype="step", lw=1.4, label="interior pos", density=True)
    ax[1].hist(sd[bmask], bins=40, histtype="step", lw=1.4, label="border pos", density=True, color="#c44e52")
    ax[1].set(xlabel="across-draw activation std", ylabel="density",
              title="border vs interior output positions")
    ax[1].legend(fontsize=8)
    fig.tight_layout(); plt.show()

    from scipy.stats import spearmanr
    rho, p = spearmanr(pv, sd)
    pos_rows.append({
        "run": RUN_DISPLAY_NAME[label],
        "spearman(patch_var, act_std)": rho,
        "act_std border pos (mean)": float(sd[bmask].mean()),
        "act_std interior pos (mean)": float(sd[~bmask].mean()),
        "ratio interior/border": float(sd[~bmask].mean() / max(sd[bmask].mean(), 1e-12)),
        "near-zero rate border pos": float(n0[bmask].mean()),
        "near-zero rate interior pos": float(n0[~bmask].mean()),
        "n border pos": int(bmask.sum()),
    })

display(pd.DataFrame(pos_rows).set_index("run").T.style.format("{:.4f}"))
print("Read: if 'ratio interior/border' >> 1, posterior activation uncertainty is "
      "concentrated on interior (signal-bearing) positions and the always-black "
      "border is left quiet -- the behaviour a sticky sampler should produce.")


### 3c. conv1 output on a blank frame vs a real digit

Push the **posterior-mean** `conv1` kernels + bias over (i) an all-zero padded
frame and (ii) one real digit, per filter. On the blank frame the output is
just the bias broadcast plus edge effects; on the digit it should light up on
strokes and stay flat on the border.


In [ ]:
_digit_idx = int(_sample_idx[5])
_digit_img = _mnist_train[_digit_idx][0].squeeze(0)                                  # [28,28]
_digit_padded = torch.nn.functional.pad(_digit_img, [2, 2, 2, 2]).view(1, 1, 32, 32)
_blank_padded = torch.zeros(1, 1, 32, 32)

for label, r in runs.items():
    ldf = layer_idx_dfs[label]
    w_coords = torch.as_tensor(ldf.index[ldf["layer"] == "conv1.weight"].to_numpy())
    b_coords = torch.as_tensor(ldf.index[ldf["layer"] == "conv1.bias"].to_numpy())
    s = r["ckpt"]["samples"]
    w = s[:, w_coords].mean(0).view(6, 1, 5, 5).float()
    b = s[:, b_coords].mean(0).float()

    with torch.no_grad():
        out_blank = torch.nn.functional.conv2d(_blank_padded, w, b).squeeze(0)   # [6,28,28]
        out_digit = torch.nn.functional.conv2d(_digit_padded, w, b).squeeze(0)

    fig, axes = plt.subplots(2, 7, figsize=(14, 4.4))
    axes[0, 0].imshow(torch.zeros(28, 28), cmap="gray"); axes[0, 0].set_title("blank frame", fontsize=9)
    axes[1, 0].imshow(_digit_img, cmap="gray");          axes[1, 0].set_title("real digit", fontsize=9)
    for c in range(6):
        axes[0, c + 1].imshow(out_blank[c], cmap="viridis"); axes[0, c + 1].set_title(f"ch{c} | blank", fontsize=9)
        axes[1, c + 1].imshow(out_digit[c], cmap="viridis"); axes[1, c + 1].set_title(f"ch{c} | digit", fontsize=9)
    for a in axes.flat:
        a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f"{RUN_DISPLAY_NAME[label]}: posterior-mean conv1 output, blank vs digit", fontsize=12)
    fig.tight_layout(); plt.show()


## 4. Sparsity across the resampled path

`prune_frac` is the cold-start frozen fraction (t=0). `sparsity_frac` (saved)
is the near-zero fraction over the resampled draws. Per-draw sparsity along
the path shows whether the sampler thawed (sparsity falls) or froze further
(rises) over the 1M events.


In [ ]:
rows = []
for label, r in runs.items():
    ck = r["ckpt"]
    per_draw = (ck["samples"].abs() < 1e-8).float().mean(1)
    rows.append({
        "run": RUN_DISPLAY_NAME[label],
        "prune_frac (t=0)": ck["prune_frac"],
        "sparsity_frac (saved)": ck["sparsity_frac"],
        "per-draw sparsity min": float(per_draw.min()),
        "per-draw sparsity mean": float(per_draw.mean()),
        "per-draw sparsity max": float(per_draw.max()),
        "bound_violations": ck["bound_violations"],
        "wall h": ck["elapsed_sec"] / 3600,
    })
display(pd.DataFrame(rows).set_index("run").style.format({
    "prune_frac (t=0)": "{:.4f}", "sparsity_frac (saved)": "{:.4f}",
    "per-draw sparsity min": "{:.4f}", "per-draw sparsity mean": "{:.4f}",
    "per-draw sparsity max": "{:.4f}", "wall h": "{:.2f}",
}))

fig, ax = plt.subplots(figsize=(9, 3.6))
for label, r in runs.items():
    per_draw = (r["ckpt"]["samples"].abs() < 1e-8).float().mean(1)
    ax.plot(per_draw.numpy(), lw=0.8, label=RUN_DISPLAY_NAME[label])
for label, r in runs.items():
    ax.axhline(r["ckpt"]["prune_frac"], ls="--", lw=0.8, alpha=0.5,
               color="grey" if label == "zigzag" else "black")
ax.set(xlabel="resampled draw index", ylabel="fraction of D with |param| < 1e-8",
       title="Sparsity along the resampled path (dashed = cold-start prune_frac)", ylim=(0, 1))
ax.legend(fontsize=8)
fig.tight_layout(); plt.show()


### 4b. Never-left-zero vs moved, by layer

No per-iteration diagnostics were saved (section 1b), so no bounce/thaw event
mix. What we *can* read directly from `samples` + `cold_start_mask`:

- **never left zero** : cold-start frozen AND every draw is (near-)zero.
- **moved**           : left near-zero in at least one draw.

A layer that is almost entirely "never left zero" is a stuck / genuinely-flat
region. Biases are never cold-start frozen, so a bias in "never left zero" was
frozen dynamically during the run.


In [ ]:
mask_rows = []
by_layer_frames = {}
for label, r in runs.items():
    ck = r["ckpt"]
    s = ck["samples"]
    cold = ck["cold_start_mask"]
    ever_nonzero = (s.abs() > 1e-8).any(0)
    never_left_zero = cold & ~ever_nonzero
    moved = ever_nonzero

    ldf = layer_idx_dfs[label].copy()
    ldf["never_left_zero"] = never_left_zero.numpy()
    ldf["moved"] = moved.numpy()
    ldf["cold"] = cold.numpy()
    by_layer = ldf.groupby("layer", sort=False).agg(
        n=("layer", "size"),
        cold_frozen=("cold", "sum"),
        never_left_zero=("never_left_zero", "sum"),
        moved=("moved", "sum"),
    )
    by_layer["frac_never_moved"] = by_layer["never_left_zero"] / by_layer["n"]
    by_layer["frac_moved"] = by_layer["moved"] / by_layer["n"]
    by_layer_frames[label] = by_layer

    mask_rows.append({
        "run": RUN_DISPLAY_NAME[label], "D": int(s.shape[1]),
        "cold-start frozen": int(cold.sum()),
        "never left zero": int(never_left_zero.sum()),
        "moved": int(moved.sum()),
        "frac never moved": float(never_left_zero.float().mean()),
    })

    print(f"[{RUN_DISPLAY_NAME[label]}]")
    display(by_layer.style.format({"frac_never_moved": "{:.4f}", "frac_moved": "{:.4f}"})
            .background_gradient(subset=["frac_never_moved"], cmap="Reds", vmin=0, vmax=1))

display(pd.DataFrame(mask_rows).set_index("run").style.format({"frac never moved": "{:.4f}"}))

fig, axes = plt.subplots(1, len(runs), figsize=(6.5 * len(runs), 3.6), squeeze=False)
for ax, (label, by_layer) in zip(axes[0], by_layer_frames.items()):
    x = np.arange(len(by_layer))
    ax.bar(x - 0.19, by_layer["frac_never_moved"], 0.38, label="never left zero", color="#c44e52")
    ax.bar(x + 0.19, by_layer["frac_moved"], 0.38, label="moved", color="#55a868")
    ax.set_xticks(x); ax.set_xticklabels(by_layer.index, rotation=30, ha="right")
    ax.set(ylabel="fraction of layer", title=RUN_DISPLAY_NAME[label], ylim=(0, 1))
    ax.legend(fontsize=8)
fig.tight_layout(); plt.show()


## 5. How far are the draws from the MAP reference?

`x_ref` in each checkpoint is the cold-start / MAP point. We also load a
MAP-reference checkpoint carrying `Sigma_inv = prior_precision + fisher_diag`,
so displacement can be expressed in **prior-std units** and compared across
layers with very different weight scales.


In [ ]:
MAP_REF_PATH = Path("results/maps/lenet_reference_N60000_steps10000.pt")
sigma_inv = None
if MAP_REF_PATH.exists():
    mref = torch.load(MAP_REF_PATH, map_location="cpu", weights_only=False)
    if int(mref["x_ref"].shape[0]) == next(iter(runs.values()))["D"]:
        sigma_inv = mref["Sigma_inv"].cpu()
        print(f"loaded Sigma_inv from {MAP_REF_PATH}  (source={mref['sigma_inv_source']})")
    else:
        print(f"{MAP_REF_PATH}: D mismatch, ignoring")
else:
    print(f"{MAP_REF_PATH} not found -- displacement shown in raw units")

per_layer_disp = []
disp_by_coord = {}
for label, r in runs.items():
    ck = r["ckpt"]
    x_ref = ck["x_ref"].cpu()
    s = ck["samples"].cpu()
    abs_disp = (s - x_ref).abs()  # [10000, D]
    if sigma_inv is not None:
        prior_std = sigma_inv.clamp(min=1e-12).rsqrt()
        scaled = abs_disp / prior_std
        unit = "prior std"
    else:
        scaled = abs_disp
        unit = "raw"
    mean_disp = scaled.mean(0)  # [D]
    disp_by_coord[label] = mean_disp

    ldf = layer_idx_dfs[label].copy()
    ldf["mean_disp"] = mean_disp.numpy()
    bl = ldf.groupby("layer", sort=False)["mean_disp"].agg(["mean", "median", "max"])
    bl.columns = pd.MultiIndex.from_product([[f"{RUN_DISPLAY_NAME[label]} ({unit})"], bl.columns])
    per_layer_disp.append(bl)

display(pd.concat(per_layer_disp, axis=1).style.format("{:.4f}").background_gradient(cmap="viridis", axis=None))

fig, ax = plt.subplots(figsize=(9, 3.6))
for label, r in runs.items():
    md_ = disp_by_coord[label]
    ax.hist(md_.numpy(), bins=120, histtype="step", lw=1.3, label=RUN_DISPLAY_NAME[label])
ax.set(xlabel=f"per-coordinate mean |sample - x_ref|  ({unit})", ylabel="count",
       title="Displacement from MAP, per coordinate", yscale="log")
ax.legend(fontsize=8)
fig.tight_layout(); plt.show()


## 6. Posterior uncertainty in `conv1` activations on a real digit

Kept verbatim from `sticky_cnn_eval.ipynb` section 6: push `N_DRAWS_SHOWN`
actual posterior draws of the `conv1` weights + bias through `conv1` on one
real digit, and look at the mean and std of the resulting 6 activation maps.
The claim being checked is that the across-draw std concentrates on the
digit's strokes and stays near zero on the background.


In [ ]:
LAYER_SHAPES = {
    "lenet5": LENET5_SHAPES,
}
D_TO_ARCH = {sum(int(np.prod(s)) for _, s in v): k for k, v in LAYER_SHAPES.items()}

N_DRAWS_SHOWN = 1000
_digit_idx = int(_sample_idx[5])
_digit_img = _mnist_train[_digit_idx][0].squeeze(0)  # [28, 28]
_digit_padded = torch.nn.functional.pad(_digit_img, [2, 2, 2, 2]).unsqueeze(0).unsqueeze(0)  # [1,1,32,32]

lenet_runs = [(label, r) for label, r in runs.items() if D_TO_ARCH.get(r["D"]) == "lenet5"]
skipped = [label for label, r in runs.items() if D_TO_ARCH.get(r["D"]) != "lenet5"]
for label in skipped:
    print(f"[{label}] not LeNet5 -- conv1 forward pass is LeNet5-specific, skip")

n_runs = len(lenet_runs)
fig, axes = plt.subplots(2 * n_runs, 7, figsize=(14, 4.2 * n_runs))
if n_runs == 1:
    axes = axes.reshape(2, 7)

for row_block, (label, r) in enumerate(lenet_runs):
    ldf = layer_idx_dfs[label]
    w_coords = torch.as_tensor(ldf[ldf["layer"] == "conv1.weight"].index.to_numpy())
    b_coords = torch.as_tensor(ldf[ldf["layer"] == "conv1.bias"].index.to_numpy())

    samples = r["ckpt"]["samples"].cpu()
    n_draws = min(N_DRAWS_SHOWN, samples.shape[0])
    draw_idx = torch.randperm(samples.shape[0])[:n_draws]

    acts = []
    with torch.no_grad():
        for i in draw_idx:
            w = samples[i, w_coords].view(6, 1, 5, 5).float()
            b = samples[i, b_coords].float()
            acts.append(torch.nn.functional.conv2d(_digit_padded.float(), w, b).squeeze(0))  # [6,28,28]
    acts = torch.stack(acts)  # [n_draws, 6, 28, 28]

    mean_act = acts.mean(dim=0)
    std_act = acts.std(dim=0)

    mean_row, std_row = axes[2 * row_block], axes[2 * row_block + 1]
    display_name = RUN_DISPLAY_NAME.get(label, label)

    mean_row[0].imshow(_digit_img, cmap="gray")
    mean_row[0].set_title("input digit", fontsize=9)
    std_row[0].axis("off")
    std_row[0].text(0.5, 0.5, display_name, fontsize=11, fontweight="semibold",
                    ha="center", va="center", transform=std_row[0].transAxes)
    for c in range(6):
        mean_row[c + 1].imshow(mean_act[c], cmap="viridis")
        mean_row[c + 1].set_title(f"ch{c} mean act.", fontsize=9)
        std_row[c + 1].imshow(std_act[c], cmap="viridis")
        std_row[c + 1].set_title(f"ch{c} std across draws", fontsize=9)

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle("Posterior uncertainty in conv1 concentrates on the digit's strokes, not the background", fontsize=12)
fig.tight_layout()
plt.show()


## 7. Predictive accuracy and uncertainty

Rebuild the target (MNIST subset + LeNet5, matching
`fast_mnist_cnn.py`), push a subsample of draws plus `x_ref` through the
network, and compare posterior-averaged predictions against the single MAP
point. The gap between them is what the sampler adds over a point estimate.

This is the most expensive cell (real CNN forward passes).


In [ ]:
from sazz.gpu_friendly.models.neural_networks import LeNet5
from sazz.gpu_friendly.models.model import to_param_dict
from sazz.gpu_friendly.scripts.fast_mnist_cnn import load_mnist_subset, N_TRAIN, N_TEST, BASE_SEED, DATA_DIR
from sazz.utils.metrics import classification_metrics

N_UNCERTAINTY_DRAWS = 300

print("Loading MNIST test set (same seed/split as the run) ...")
data = load_mnist_subset(N_TRAIN, N_TEST, BASE_SEED, DATA_DIR, dtype=torch.float64, device="cpu")
X_test, y_test = data["X_test"], data["y_test"]

perf_rows = []
predictive_probs = {}
for label, r in runs.items():
    ck = r["ckpt"]
    module = LeNet5(activation=ck["activation"], pool=ck["pool"]).to(dtype=torch.float64)
    names = [n for n, _ in module.named_parameters()]
    shapes = [p.shape for _, p in module.named_parameters()]
    param_dict_fn = to_param_dict(names, shapes)

    x_ref = ck["x_ref"].to(dtype=torch.float64)
    samples = ck["samples"].to(dtype=torch.float64)
    idx = torch.randperm(samples.shape[0])[:min(N_UNCERTAINTY_DRAWS, samples.shape[0])]
    sub = samples[idx]

    with torch.no_grad():
        map_logits = torch.func.functional_call(module, param_dict_fn(x_ref), (X_test,))
        map_probs = torch.softmax(map_logits, dim=-1)
        draw_probs = []
        for beta in sub:
            logits = torch.func.functional_call(module, param_dict_fn(beta), (X_test,))
            draw_probs.append(torch.softmax(logits, dim=-1))
        draw_probs = torch.stack(draw_probs)
        mean_probs = draw_probs.mean(0)

    predictive_probs[label] = {"map": map_probs, "posterior_draws": draw_probs, "posterior_mean": mean_probs}
    map_metrics = classification_metrics(y_test, map_probs)
    post_metrics = classification_metrics(y_test, mean_probs)
    p_true_per_draw = draw_probs.gather(-1, y_test.long().view(1, -1, 1).expand(draw_probs.shape[0], -1, 1)).squeeze(-1)
    disagreement = p_true_per_draw.std(dim=0).mean().item()

    perf_rows.append({
        "run": RUN_DISPLAY_NAME[label],
        "map accuracy": map_metrics["accuracy"], "map log_lik": map_metrics["log_lik"],
        "map entropy": map_metrics["entropy"],
        "posterior accuracy": post_metrics["accuracy"], "posterior log_lik": post_metrics["log_lik"],
        "posterior entropy": post_metrics["entropy"],
        "ckpt test_accuracy": ck.get("test_accuracy"),
        "mean draw-to-draw P(true class) std": disagreement,
    })

perf_df = pd.DataFrame(perf_rows).set_index("run")
display(perf_df.style.format("{:.4f}"))

print(
    "Read: if 'posterior entropy' > 'map entropy' at matched accuracy, the sampler is\n"
    "adding real epistemic-uncertainty signal. 'mean draw-to-draw P(true class) std'\n"
    "near 0 means the draws behave like near-duplicates of x_ref on the test set."
)


In [ ]:
# Reliability curve + entropy split for correct vs wrong predictions.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for label in runs:
    mean_probs = predictive_probs[label]["posterior_mean"]
    conf, pred = mean_probs.max(-1)
    correct = (pred == y_test.long())
    bins = torch.linspace(0, 1, 11)
    xs, accs = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf >= lo) & (conf < hi)
        if m.any():
            xs.append(((lo + hi) / 2).item())
            accs.append(correct[m].float().mean().item())
    axes[0].plot(xs, accs, marker="o", label=RUN_DISPLAY_NAME[label])

    ent = -(mean_probs.clamp_min(1e-12) * mean_probs.clamp_min(1e-12).log()).sum(-1)
    axes[1].hist(ent[correct].numpy(), bins=40, histtype="step", lw=1.4,
                 label=f"{RUN_DISPLAY_NAME[label]} correct")
    axes[1].hist(ent[~correct].numpy(), bins=40, histtype="step", lw=1.4, ls="--",
                 label=f"{RUN_DISPLAY_NAME[label]} wrong")

axes[0].plot([0, 1], [0, 1], "k:", lw=1)
axes[0].set(xlabel="posterior-mean confidence", ylabel="empirical accuracy",
            title="Reliability (posterior-averaged)")
axes[0].legend(fontsize=8)
axes[1].set(xlabel="predictive entropy", ylabel="count", title="Entropy: correct vs wrong", yscale="log")
axes[1].legend(fontsize=7)
fig.tight_layout(); plt.show()


## 8. Takeaways

Fill in after running -- the questions this notebook was built to answer:

1. **Files**: both checkpoints are `dict`s with `samples [10000, 61706]`,
   `x_ref`, `cold_start_mask`, scalar summaries, and `grid_t_max_log`.
   `diagnostics` is `None` and there is no chunk dir, so no per-iteration
   event mix / per-coordinate bounce attribution.
2. **conv1 & the black border** (sec 3): does the exact-zero rate rise and the
   per-draw std fall as the receptive pixel variance drops? Do border taps
   carry the zeros?
3. **Sparsity** (sec 4): did the path thaw or freeze relative to
   `prune_frac`?
4. **Displacement** (sec 5): are the draws exploring around `x_ref` or sitting
   on it, and which layers move most in prior-std units?
5. **Predictions** (sec 6-7): is the posterior-averaged prediction more
   usefully uncertain than the MAP point at the same accuracy?
